In [ ]:
import pandas as pd
import numpy as np
import kagglehub
import random
import torch
import torch.nn as nn
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

from transformers import AutoTokenizer, AutoModel, AutoConfig
from torch.utils.data import Dataset, DataLoader
from transformers.utils import logging

In [ ]:
class cfg:
    competition = 'learning-agency-lab-automated-essay-scoring-2'
    checkpoint = 'microsoft/deberta-v3-small'
    tokenizer = None
    max_length = 512
    batch_size = 16
    epochs = 10
    learning_rate = 5e-5
    weight_decay = 0.01
    task = 'regression'

In [ ]:
#Quadratic Weighted Kappa score
def get_score(y_true, y_pred):
    return cohen_kappa_score(
        y_true,
        y_pred,
        weights="quadratic"
    )

In [ ]:
#Sets random to 42
def seed_everything(seed: int=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    
seed_everything(42)

In [ ]:
path = kagglehub.competition_download(cfg.competition)

In [ ]:
train_df = pd.read_csv(f'{path}/train.csv')

In [ ]:
#Label y starting from 0 so classifier works
if train_df['score'].min() == 1:
    train_df['score'] = train_df['score'] - 1

In [ ]:
#Cross Validation
train_df['fold'] = -1
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, valid_idx) in enumerate(skf.split(train_df, train_df['score'])):
    train_df.loc[valid_idx, 'fold'] = fold

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.checkpoint)

In [ ]:
class EssayDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=cfg.max_length):
        self.X = data['full_text']
        if 'score' in data.columns:
            self.y = data['score']
        else:
            self.y = None
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        tokens = self.tokenizer(
            text = self.X.iloc[idx],
            max_length=self.max_length,
            truncation = True,
            padding = 'max_length',
            return_tensors = 'pt'
        )
        item = {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
        }
        if self.y is not None:
            item['labels'] = torch.tensor(self.y.iloc[idx], dtype=torch.float)
        return item


In [ ]:
#MeanPooling code taken from online resources
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()

    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_embeddings/sum_mask
        return mean_embeddings

In [ ]:
class EssayModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained(cfg.checkpoint)
        self.mp = MeanPooling()
        self.classifier = nn.Linear(self.model.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids = input_ids, attention_mask = attention_mask)
        # #CLS Pooling
        # cls_output = output.last_hidden_state[:, 0, :].float()
        # logits = self.classifier(cls_output)
        #Mean Pooling
        mp_output = self.mp(output.last_hidden_state, attention_mask)
        logits = self.classifier(mp_output)
        return logits


In [ ]:
print(torch.backends.mps.is_available())
device = torch.device('mps')
#For CUDA:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#Global Debug variable for testing
DEBUG = True

In [ ]:
#Store overall best QWK:
all_oof_preds = []
all_oof_labels = []

for fold in range(5):
    valid_fold = train_df[train_df['fold']==fold] 

    valid_dataset = EssayDataset(valid_fold, tokenizer)
    valid_loader = DataLoader(valid_dataset, batch_size=cfg.batch_size, shuffle=False)

    #Create model object
    logging.set_verbosity_error()
    deberta_model = EssayModel().to(device).float()

    deberta_model.load_state_dict(
        torch.load(f'best_deberta_model_fold_{fold}.pth', map_location=device)
    )
    deberta_model.eval()

    with torch.no_grad():
        for batch in valid_loader:
            logits = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device)).squeeze(-1)
            all_oof_preds.extend(logits.cpu().numpy())
            all_oof_labels.extend(batch['labels'].numpy())

In [ ]:
def find_thresholds(true, pred, steps=50):
    xs = [[], [], [], [], []]
    ys = [[], [], [], [], []]

    threshold = [0.5, 1.5, 2.5, 3.5, 4.5]

    pred2 = pd.cut(
        pred,
        [-np.inf] + threshold + [np.inf],
        labels=[0, 1, 2, 3, 4, 5]
    ).astype("int32")

    best = cohen_kappa_score(true, pred2, weights="quadratic")

    for k in range(5):
        for sign in [1, -1]:
            v = threshold[k]
            threshold2 = threshold.copy()
            stop = 0

            while stop < steps:
                v += sign * 0.001
                threshold2[k] = v

                pred2 = pd.cut(
                    pred,
                    [-np.inf] + threshold2 + [np.inf],
                    labels=[0, 1, 2, 3, 4, 5]
                ).astype("int32")

                metric = cohen_kappa_score(true, pred2, weights="quadratic")

                xs[k].append(v)
                ys[k].append(metric)

                if metric <= best:
                    stop += 1
                else:
                    stop = 0
                    best = metric
                    threshold = threshold2.copy()

    threshold = [np.round(t, 3) for t in threshold]
    return best, threshold, xs, ys

In [ ]:
best, thresholds, xs, ys = find_thresholds(
    np.array(all_oof_labels).astype(int),
    np.array(all_oof_preds),
    steps=500
)

print("Best thresholds are:", thresholds)
print("=> achieve Overall CV QWK score =", best)